# Common setup code

In [229]:
from collections.abc import Callable
from copy import deepcopy

""" Matrices definitions """
# Additional smaller matrix to more easily observe patterns in transformed data
MATRIX0 = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

EXPECTED0 = [
    [7, 4, 1],
    [8, 5, 2],
    [9, 6, 3],
]

MATRIX1 = [
    [ 1,  2,  3,  4],
    [ 5,  6,  7,  8],
    [ 9, 10, 11, 12],
    [13, 14, 15, 16],
]

EXPECTED1 = [
    [13,  9,  5,  1],
    [14, 10,  6,  2],
    [15, 11,  7,  3],
    [16, 12,  8,  4],
]

MATRIX2 = [
    [ 1,  2,  3,  4,  5,  6,  7],
    [ 8,  9, 10, 11, 12, 13, 14],
    [15, 16, 17, 18, 19, 20, 21],
    [22, 23, 24, 25, 26, 27, 28],
    [29, 30, 31, 32, 33, 34, 35],
    [36, 37, 38, 39, 40, 41, 42],
    [43, 44, 45, 46, 47, 48, 49],
]

EXPECTED2 = [
    [43, 36, 29, 22, 15,  8,  1],
    [44, 37, 30, 23, 16,  9,  2],
    [45, 38, 31, 24, 17, 10,  3],
    [46, 39, 32, 25, 18, 11,  4],
    [47, 40, 33, 26, 19, 12,  5],
    [48, 41, 34, 27, 20, 13,  6],
    [49, 42, 35, 28, 21, 14,  7],
]


def assess(rotation_fn: Callable[[list[list]], list[list]]):
    for testcase, expected in ((MATRIX0, EXPECTED0), (MATRIX1, EXPECTED1), (MATRIX2, EXPECTED2)):
        # Creating a temporary test case variable, so that the originals are not
        # modified when they are passed into the rotation function
        temp_testcase = deepcopy(testcase)

        # Modified to use a returned value, instead of expecting input mutation
        # Which is unable to happen with a pass-by-value like as it was before
        rotated = rotation_fn(temp_testcase)

        if rotated != expected:
            print('Testcase failed. Actual vs. Expected:')
            n = len(expected)
            for r1, r2 in zip(rotated, expected):
                print(f'{str(r1):<{4*n}} {r2}')
            print()
        else:
            print('Testcase OK!')

# Naive solution
## Notes
- The code is attempting to rotate the matrix in-place, however, it overwrites the elements
- Using the same formula, we can fix the code simply by changing its use of data structures
  - Use the return at the call site (so as to avoid pass-by-value issues)
  - Deep copy (not shallow copy / variable re-binding) the object so we do not overwrite elements

## Shortcomings
- This however, isn't rotating the matrix in-place
- Assuming `n` is the number of elements in a matrix
  - It uses O(n^2) space
    - As it stores the original matrix (n) and duplicate (n) 
  - It takes O(n^2) time
    - As it must traverse every element once


In [230]:
# Added type hinting
def rotate_in_place(matrix: list[list]):
    n = len(matrix)

    # Deep copy to avoid element overwriting
    rotated = deepcopy(matrix)

    for r in range(n):
        for c in range(n):
            rotated[r][c] = matrix[n-c-1][r]

    # Return to avoid variable binding issues
    return rotated

assess(rotate_in_place)

Testcase OK!
Testcase OK!
Testcase OK!


# More complex solution - transposition and reversal
- Initially, I begun investigating this problem by inquiring if there was a way to use simple, commonplace matrix operations to achieve this result
- One that's similar to this rotation is matrix transposition
- After some Googling and performing a sample transformation on paper, it became clear that we could combine transposition with row reversal (mirroring) to achieve an in-place variant of this algorithm
- This has the same time complexity, but requires only O(n) space

In [231]:
def transpose_and_mirror(matrix: list[list]):
    # n = side length of matrix
    # (n-1) = max element index of a given row or column
    n = len(matrix)

    # Transpose in-place
    # Only do the upper triangle of the matrix (excluding the diagonal)
    # [0, n)
    for r in range(0, n):
        # [r+1, n)
        for c in range(r + 1, n):
            tmp = matrix[r][c]
            matrix[r][c] = matrix[c][r]
            matrix[c][r] = tmp

    # Reverse rows in-place (mirror matrix around the vertical axis)
    # Only iterate one (the left) half of the matrix (excluding the middle column)
    # [0, n // 2)
      # Flooring integer division used for this
    # Alternatively, loop over rows and use the inbuilt `r.reverse()`
    for r in range(n):
        for c in range(0, n // 2):
            swap_col = (n - 1) - c
            tmp = matrix[r][c]
            matrix[r][c] = matrix[r][swap_col]
            matrix[r][swap_col] = tmp

    return matrix

assess(transpose_and_mirror)

Testcase OK!
Testcase OK!
Testcase OK!


In [232]:
print(MATRIX0)

[[1, 2, 3], [4, 5, 6], [7, 8, 9]]


# Addendum
- Another solution seen online involves rotating the matrix in "rings" around the centre
- This involves only 1 temporary variable, and so removes the O(n^2) space requirement